# 15.5 Solving network linear programs

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

All of the models that we have described in this chapter give rise to linear programs
that have a "network" property of some kind. AMPL can send these linear programs to
an LP solver and retrieve the optimal values, much as for any other class of LPs. If you
use AMPL in this way, the network structure is helpful mainly as a guide to formulating
the model and interpreting the results.

Many of the models that we have described belong as well to a more restricted class
of problems that (confusingly) are also known as "network linear programs." In modeling
terms, the variables of a network LP must represent flows on the arcs of a network,
and the constraints must be only of two types: bounds on the flows along the arcs, and
limits on flow out minus flow in at the nodes. A more technical way to say the same
thing is that each variable of a network linear program must appear in at most two constraints
(aside from lower or upper bounds on the variables), such that the variable has a
coefficient of +1 in at most one constraint, and a coefficient of -1 in at most one constraint.

"Pure" network linear programs of this restricted kind have some very strong properties
that make their use particularly desirable. So long as the supplies, demands, and

```ampl
set CITIES;
set LINKS within (CITIES cross CITIES);
set PRODS;
param supply {PRODS,CITIES} >= 0;  # amounts available at cities
param demand {PRODS,CITIES} >= 0;  # amounts required at cities
       check {p in PRODS}:
              sum {i in CITIES} supply[p,i] = sum {j in CITIES} demand[p,j];
param cost {PRODS,LINKS} >= 0;     # shipment costs/1000 packages
param capacity {PRODS,LINKS} >= 0; # max packages shipped of product
set FEEDS;
param yield {PRODS,FEEDS} >= 0;    # amounts derived from feedstocks
param limit {FEEDS,CITIES} >= 0;   # feedstocks available at cities
minimize Total_Cost;
var Feed {f in FEEDS, k in CITIES} >= 0, <= limit[f,k];
node Balance {p in PRODS, k in CITIES}:
net_out = supply[p,k] - demand[p,k]
       + sum {f in FEEDS} yield[p,f] * Feed[f,k];
arc Ship {p in PRODS, (i,j) in LINKS} >= 0, <= capacity[p,i,j],
from Balance[p,i], to Balance[p,j],
obj Total_Cost cost[p,i,j];
```

**Figure 15-14:** Multicommodity flow with side variables (netfeeds.mod).

bounds are integers, a network linear program must have an optimal solution in which all
flows are integers. Moreover, if the solver is of a kind that finds "extreme" solutions
(such as those based on the simplex method) it will always find one of the all-integer
optimal solutions. We have taken advantage of this property, without explicitly mentioning
it, in assuming that the variables in the shortest path problem and in certain assignment
problems come out to be either zero or one, and never some fraction in between.

Network linear programs can also be solved much faster than other linear programs of
comparable size, through the use of solvers that are specialized to take advantage of the
network structure. If you write your model in terms of `node` and `arc` declarations,
AMPL automatically communicates the network structure to the solver, and any special
network algorithms available in the solver can be applied automatically. On the other
hand, a network expressed algebraically using `var` and `subject to` may or may not be
recognized by the solver, and certain options may have to be set to ensure that it is recognized.
For example, when using the algebraic model of [Figure 15-4a](./15_1_minimumcost_transshipment_models.ipynb#fig-15-4a), you may see the
usual response from the general LP algorithm:

```ampl
ampl: model net3.mod; data net3.dat; solve;
CPLEX 8.0.0: optimal solution; objective 1819
1 dual simplex iterations (0 in phase I)
```

But when using the equivalent `node` and `arc` formulation of Figure 15-11, you may get a
somewhat different response to reflect the application of a special network LP algorithm:

```ampl
ampl: model net3node.mod
ampl: data net3.dat
ampl: solve;
CPLEX 8.0.0: optimal solution; objective 1819
Network extractor found 7 nodes and 7 arcs.

7 network simplex iterations.
0 simplex iterations (0 in phase I)
```

To determine how your favorite solver behaves in this situation, consult the solverspecific
documentation that is supplied with your AMPL installation.

Because network linear programs are much easier to solve, especially with integer
data, the success of a large-scale application may depend on whether a pure network formulation
is possible. In the case of the multicommodity flow model of Figure 15-13, for
example, the joint capacity constraints disrupt the network structure — they represent a
third constraint in which each variable figures — but their presence cannot be avoided in
a correct representation of the problem. Multicommodity flow problems thus do not necessarily
have integer solutions, and are generally much harder to solve than singlecommodity
flow problems of comparable size.

In some cases, a judicious reformulation can turn what appears to be a more general
model into a pure network model. Consider, for instance, a generalization of [Figure 15-10](./15_3_declaring_network_models_by_node_and_arc.ipynb#fig-15-10)
in which capacities are defined at the nodes as well as along the arcs:

```ampl
param city_cap {CITIES} >= 0;
param link_cap {LINKS} >= 0;
```

The `arc` capacities represent, as before, upper limits on the shipments between cities. The
`node` capacities limit the throughput, or total flow handled at a city, which may be written
as the supply at the city plus the sum of the flows in, or equivalently as the demand at the
city plus the sum of the flows out. Using the former, we arrive at the following constraint:

```ampl
subject to through_limit {k in CITIES}:
       supply[k] + sum {(i,k) in LINKS} Ship[i,k] <= node_cap[k];
```

Viewed in this way, the throughput limit is another example of a "side constraint" that
disrupts the network structure by adding a third coefficient for each variable. But we can
achieve the same effect without a side constraint, by using two nodes to represent each
city; one receives flow into a city plus any supply, and the other sends flow out of a city
plus any demand:

```ampl
node Supply {k in CITIES}: net_out = supply[k];
node Demand {k in CITIES}: net_in = demand[k];
```

A shipment link between cities `i` and `j` is represented by an `arc` that connects the `node`
Demand[i] to `node` Supply[j]:

```ampl
set CITIES;
set LINKS within (CITIES cross CITIES);
param supply {CITIES} >= 0;   # amounts available at cities
param demand {CITIES} >= 0;   # amounts required at cities
check: sum {i in CITIES} supply[i] = sum {j in CITIES} demand[j];
param cost {LINKS} >= 0;      # shipment costs per ton
param city_cap {CITIES} >= 0; # max throughput at cities
param link_cap {LINKS} >= 0; # max shipment over links
minimize Total_Cost;
node Supply {k in CITIES}: net_out = supply[k];
node Demand {k in CITIES}: net_in = demand[k];
arc Ship {(i,j) in LINKS} >= 0, <= link_cap[i,j],
from Demand[i], to Supply[j], obj Total_Cost cost[i,j];
arc Through {k in CITIES} >= 0, <= city_cap[k],
from Supply[k], to Demand[k];
```

**Figure 15-15:** Transshipment model with `node` capacities (netthru.mod).

```ampl
arc Ship {(i,j) in LINKS} >= 0, <= link_cap[i,j],
       from Demand[i], to Supply[j], obj Total_Cost cost[i,j];
```

The throughput at city k is represented by a new kind of `arc`, from Supply[k] to
Demand[k]:

```ampl
arc Through {k in cities} >= 0, <= city_cap[k],
       from Supply[k], to Demand[k];
```

The throughput limit is now represented by an upper bound on this arc's flow, rather than
by a side constraint, and the network structure of the model is preserved. A complete listing
appears in Figure 15-15.

The preceding example exhibits an additional advantage of using the `node` and `arc`
declarations when developing a network model. If you use only `node` and `arc` in their
simple forms — no variables in the `node` conditions, and no optional factors in the from
and to phrases — your model is guaranteed to give rise only to pure network linear
programs. By contrast, if you use `var` and `subject to`, it is your responsibility to ensure
that the resulting linear program has the necessary network structure.

Some of the preceding comments can be extended to "generalized network" linear
programs in which each variable still figures in at most two constraints, but not necessarily
with coefficients of +1 and -1. We have seen examples of generalized networks in the
cases where there is a loss of flow or change of units on the arcs. Generalized network
LPs do not necessarily have integer optimal solutions, but fast algorithms for them do
exist. A solver that promises a "network" algorithm may or may not have an extension
to generalized networks; `check` the solver-specific documentation before you make any
assumptions.
**Bibliography**

Ravindra K. Ahuja, Thomas L. Magnanti and James B. Orlin, Network Flows: Theory, Algorithms,
and Applications. Prentice-Hall (Englewood Cliffs, NJ, 1993).

Dimitri P. Bertsekas Network Optimization: Continuous and Discrete Models. Athena Scientific
(Princeton, NJ, 1998).

L. R. Ford, Jr. and D. R. Fulkerson, Flows in Networks. Princeton University Press (Princeton, NJ, 1962).
A highly influential survey of network linear programming and related topics, which stimulated
much subsequent study.

Fred Glover, Darwin Klingman and Nancy V. Phillips, Network Models in Optimization and their
Applications in Practice. John Wiley & Sons (New York, 1992).

Walter Jacobs, "The Caterer Problem." Naval Research Logistics Quarterly 1 (1954) pp. 154-165.
The origin of the network problem described in Exercise 15-8.

Katta G. Murty, Network Programming. Prentice-Hall (Englewood Cliffs, NJ, 1992).